# LiCSBAS batch processing — python & shell

This notebook demonstrates the two ways to run a full LiCSBAS processing chain with the
**v3 layout** (pip-installable `licsbas` package):

1. **Python batch** — `licsbas.batch` (recommended): a plain-Python config object
   (`BatchConfig`) that maps 1:1 to the old `batch_LiCSBAS.sh` settings; the steps are
   executed *in-process* through `licsbas.steps.*`, and a PNG flowchart of the chain
   can be produced with `save_flowchart()`.
2. **Shell batch** — the legacy `./batch_LiCSBAS.sh` invocation, which in v3 is a thin
   wrapper around `python -m licsbas.batch`, so all CLI flags (`--sbovl`, `--reunw`,
   `--check-only`, `--flowchart`, ...) work exactly as expected.

All cells below run in **dry-run mode** by default (nothing is downloaded / computed):
set `DRY_RUN = False` in the first code cell and put this notebook next to your
`GEOC`/`TS_*` products to process real data.


In [ ]:
import os, sys

# make the repo (or installed package) importable when running from a checkout
_REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.isdir(os.path.join(_REPO, "licsbas")):
    sys.path.insert(0, _REPO)
import licsbas
print("using licsbas package:", licsbas.__file__)

# ---- toggle: True = only show what would be done (safe default) ----
DRY_RUN = True

## 1. The standard chain (historical `batch_LiCSBAS.sh` defaults)

Step 01 (get geotiff) → 02 (ml prep) → 11 (check) → 12 (loop closure) → 13 (SB inv)
→ 14 (velocity/std) → 15 (mask ts) → 16 (filter ts).
This is exactly what the old batch script did out of the box — and what
`examples/example_1_standard.py` runs.


In [ ]:
from licsbas.batch import BatchConfig, build_pipeline, run_batch, save_flowchart
cfg = BatchConfig(start_step=1, end_step=16, nlook=1)
for s in build_pipeline(cfg):
    print(f"{s.key:>6}  {s.name:<34} {' '.join(s.argv)}")

### Flowchart of the chain

Rendered with the `graphviz` `dot` binary (install the system `graphviz` package;
no extra python dependency needed). Optional steps 03-06 are shown greyed out
because they are off by default.


In [ ]:
png = save_flowchart(cfg, "flowchart_standard.png",
                     title="LiCSBAS standard processing chain")
print("written:", png)
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass  # e.g. running outside jupyter: just open the file

## 2. Re-unwrapping chain via `LiCSBAS02to05_unwrap.py`

The *updated* pipeline (`run_reunwrapping="y"` in the old script): step 01 also
downloads pha+ml, then steps 02-05 are replaced by the single re-unwrapping script
with its `p02to05_*` options (`--gacos`, `--hgtcorr`, `--cascade`, `--filter`,
`--thres`, `-g` geo clip box).


In [ ]:
cfg_reunw = BatchConfig(
    start_step=1, end_step=16, nlook=20,
    run_reunw=True,                       # <- uses LiCSBAS02to05_unwrap.py
    p02to05_hgtcorr=True,                 # recommended for high, varying terrain
    # p02to05_gacos=True, p02to05_cascade=True,
    # p02to05_filter="gold", p02to05_thres="0.35",
    # p02to05_cliparea_geo="130.11/131.12/34.34/34.6",
    p12_multi_prime=True, p12_nullify=False,
)
png = save_flowchart(cfg_reunw, "flowchart_reunw_02to05.png",
                     title="LiCSBAS re-unwrapping chain via 02to05")
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass

## 3. SBOI chain (`--sbovl` options)

`sbovl=True` switches on the sbovl chain: `--sbovl` is passed to steps
01/02/11/13/15/16, step 120 (automated reference choice) is used, and step 12 is
skipped (sbovl data is referenced differently). With `sbovl_abs=True` the
referencing is closed and absolute SET/Iono corrections are added via
`LiCSBAS131_boi_absolute.py`.


In [ ]:
cfg_sbovl = BatchConfig(
    start_step=1, end_step=16, nlook=10,
    sbovl=True,                           # <- the --sbovl chain
    # sbovl_abs=True,                     # SBOI absolute: close referencing + ICC/ESD + abs corr
    p01_get_iono=True, p01_get_tides=True,
    # p131_tide=True, p131_iono=True,     # post-13 corrections (LiCSBAS131_corrections)
    p120_use=True,
)
png = save_flowchart(cfg_sbovl, "flowchart_sbovl.png",
                     title="LiCSBAS SBOI chain (--sbovl options)")
try:
    from IPython.display import Image, display
    display(Image(filename=png))
except ImportError:
    pass

## 4. Executing a chain

`run_batch(cfg, dry_run=True)` prints the commands (like the old `check_only="y"`);
`run_batch(cfg, dry_run=False)` executes the steps in-process through
`licsbas.steps.*` (a step-120 failure only warns and continues — same as the
old script).


In [ ]:
rc = run_batch(cfg, dry_run=DRY_RUN)
print("exit code:", rc)

## 5. Shell batch (legacy invocation)

From a checkout (or, after `pip install .`, from anywhere) the old command line
still works — `batch_LiCSBAS.sh` forwards to the python batch:

```bash
source bashrc_LiCSBAS.sh          # adds bin/ to PATH (legacy step wrappers)
./batch_LiCSBAS.sh --help         # all CLI options
./batch_LiCSBAS.sh --check-only              # print the standard chain only
./batch_LiCSBAS.sh --start 11 --end 16       # continue an existing product set
./batch_LiCSBAS.sh --sbovl                   # SBOI chain (sbovl=True)
./batch_LiCSBAS.sh --sbovl-abs               # SBOI absolute chain
./batch_LiCSBAS.sh --reunw                   # 02-05 via LiCSBAS02to05_unwrap.py
./batch_LiCSBAS.sh --flowchart fc.png        # also write the PNG flowchart
```

or, equivalently, as the installed console entry point:

```bash
licsbas-batch --check-only
```


## 6. Ready-made example scripts

| script | chain | flowchart |
|--------|-------|-----------|
| `examples/example_1_standard.py` | standard (old defaults) | `flowchart_standard.png` |
| `examples/example_2_reunw_02to05.py` | 02to05 re-unwrapping | `flowchart_reunw_02to05.png` |
| `examples/example_3_sbovl.py` | SBOI `--sbovl` | `flowchart_sbovl.png` |

```bash
python3 examples/example_1_standard.py            # dry run + PNG (default)
python3 examples/example_1_standard.py --execute  # really run it
```
